# Ejecutar Streamlit desde Google Colab con ngrok

Este cuaderno arranca una aplicación de Streamlit dentro de Colab y la expone
a internet mediante un túnel de ngrok.

## Por qué hace falta ngrok

Cuando ejecutas `streamlit run` en **tu ordenador**, el servidor queda en
`localhost:8501` y tu navegador puede acceder porque está en la misma máquina.

En **Colab** no. Tu cuaderno se ejecuta en un servidor de Google, y ese
`localhost` se refiere a la máquina de Google, no a la tuya. Además, esa máquina
no expone sus puertos a internet.

ngrok resuelve esto abriendo un **túnel inverso**: un agente dentro de Colab
inicia una conexión *saliente* hacia los servidores de ngrok (que el cortafuegos
sí permite), y a partir de ahí ngrok reenvía a tu aplicación las visitas que
lleguen a una URL pública.

> **Importante:** esto NO es un despliegue. La URL muere cuando cierras Colab.
> Para publicar de forma permanente, consulta el apartado 9 del cuaderno de despliegue de streamlit (GitHub + Streamlit Community Cloud).


## Paso 1. Obtener el token de ngrok

1. Regístrate gratis en [ngrok.com](https://ngrok.com) (puedes usar Google o GitHub).
2. Ve a **Getting Started → Your Authtoken**, o directamente a
   [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
3. Copia el token (una cadena larga tipo `2abcDEF...`).

**Guárdalo en los Secrets de Colab** en lugar de escribirlo en una celda:
icono de la **llave 🔑** en el panel izquierdo → **Add new secret** →
nombre `NGROK_TOKEN`, valor tu token → activa *Notebook access*.


In [ ]:
!pip install -q streamlit pyngrok
print("Instalación completada")

## Paso 2. Escribir la aplicación

La magia `%%writefile` guarda el contenido de la celda en un fichero `.py`.
Streamlit necesita un fichero, no celdas de cuaderno.

Detalles:
- `%%writefile` debe ser la **primera línea** de la celda, sin nada delante.
- El contenido **no se ejecuta**, solo se escribe. Los errores de Python
  aparecerán después, al arrancar la aplicación.
- Al reejecutar la celda, el fichero se sobrescribe.


In [ ]:
%%writefile app.py
import matplotlib
matplotlib.use("Agg")

import numpy as np
import pandas as pd
import streamlit as st
from matplotlib.figure import Figure
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Debe ser la PRIMERA llamada st.* del script
st.set_page_config(page_title="Clasificador Iris", page_icon="🌸", layout="wide")


# @st.cache_resource: el script se reejecuta en cada interacción.
# Sin este decorador, el modelo se reentrenaría cada vez que el usuario
# moviera un slider.
@st.cache_resource
def cargar_modelo():
    iris = load_iris()
    X, y = iris.data, iris.target
    modelo = RandomForestClassifier(n_estimators=100, random_state=42)
    scores = cross_val_score(modelo, X, y, cv=5)
    modelo.fit(X, y)
    meta = {
        "clases": [str(c) for c in iris.target_names],
        "features": [str(f) for f in iris.feature_names],
        "medias": {
            str(n): X[y == i].mean(axis=0).round(3).tolist()
            for i, n in enumerate(iris.target_names)
        },
        "acc": float(scores.mean()),
        "std": float(scores.std()),
    }
    return modelo, meta


modelo, meta = cargar_modelo()
CLASES = meta["clases"]

st.title("🌸 Clasificador de flores Iris")
st.caption(
    f"Random Forest (100 árboles) · Validación cruzada: "
    f"{meta['acc']:.3f} ± {meta['std']:.3f}"
)

# --- Barra lateral: configuración ---
with st.sidebar:
    st.header("Medidas de entrada")

    ejemplos = {
        "Personalizado": [5.8, 3.0, 3.8, 1.2],
        "Setosa típica": [5.1, 3.5, 1.4, 0.2],
        "Versicolor típica": [6.7, 3.1, 4.7, 1.5],
        "Virginica típica": [6.3, 3.3, 6.0, 2.5],
    }
    eleccion = st.selectbox("Ejemplos predefinidos", list(ejemplos.keys()))
    base = ejemplos[eleccion]

    sl = st.slider("Sepal length (cm)", 4.0, 8.0, base[0], 0.1)
    sw = st.slider("Sepal width (cm)", 2.0, 4.5, base[1], 0.1)
    pl = st.slider("Petal length (cm)", 1.0, 7.0, base[2], 0.1)
    pw = st.slider("Petal width (cm)", 0.1, 2.5, base[3], 0.1)

    st.divider()
    st.caption("Modelo entrenado con 150 muestras. Demostración didáctica.")

medidas = [sl, sw, pl, pw]

# --- Predicción ---
probs = modelo.predict_proba(np.array([medidas]))[0]
idx = int(modelo.predict(np.array([medidas]))[0])
confianza = float(probs[idx])

c1, c2, c3 = st.columns(3)
c1.metric("Predicción", CLASES[idx].capitalize())
c2.metric("Confianza", f"{confianza:.1%}")
c3.metric("Segunda opción", CLASES[int(np.argsort(probs)[-2])])

if confianza < 0.60:
    st.warning("Confianza baja: estas medidas caen en una zona ambigua.")

st.divider()

izq, der = st.columns([3, 2])

with izq:
    st.subheader("Distribución de probabilidades")
    # Figure en lugar de pyplot: evita acumular figuras en memoria
    fig = Figure(figsize=(6, 3))
    ax = fig.subplots()
    colores = ["#2563eb" if i == idx else "#93c5fd" for i in range(len(probs))]
    barras = ax.bar(CLASES, probs, color=colores, edgecolor="white", linewidth=1.5)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Probabilidad")
    for b, p in zip(barras, probs):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02,
                f"{p:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    st.pyplot(fig)

with der:
    st.subheader("Detalle")
    st.dataframe(
        pd.DataFrame({"Especie": CLASES, "Probabilidad": probs.round(4)}),
        hide_index=True, width="stretch",
    )

with st.expander("Comparar con las medias de cada especie"):
    filas = [["-> Tu muestra"] + [round(v, 2) for v in medidas]]
    for nombre, m in meta["medias"].items():
        filas.append([f"media {nombre}"] + m)
    st.dataframe(
        pd.DataFrame(filas, columns=["Muestra"] + meta["features"]),
        hide_index=True, width="stretch",
    )

resultado = pd.DataFrame({
    "variable": meta["features"] + ["prediccion", "confianza"],
    "valor": [str(v) for v in medidas] + [CLASES[idx], f"{confianza:.4f}"],
})
st.download_button(
    "Descargar resultado (CSV)",
    data=resultado.to_csv(index=False).encode("utf-8"),
    file_name="prediccion_iris.csv",
    mime="text/csv",
)

## Paso 3. Configurar el token de ngrok

Lee el token desde los Secrets de Colab. Si no los has configurado, usa la
celda alternativa que viene después.


In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
print("Token configurado correctamente")

**Alternativa sin Secrets** (menos segura: el token queda escrito en el
cuaderno). Descomenta y pega tu token si prefieres esta vía:


In [ ]:
# from pyngrok import ngrok
# ngrok.set_auth_token("PEGA_AQUI_TU_TOKEN")

## Paso 4. Arrancar Streamlit y abrir el túnel

Al ejecutar esta celda obtendrás una URL pública.

La primera vez que la abras, ngrok muestra una **página de advertencia**
con un botón **"Visit Site"**. Púlsalo para llegar a la aplicación.


In [ ]:
import subprocess
import time

from pyngrok import ngrok

# 1. Cerrar túneles anteriores.
#    La cuenta gratuita solo permite UN túnel simultáneo, así que sin esto
#    la celda fallaría al reejecutarla.
ngrok.kill()

# 2. Arrancar Streamlit en segundo plano.
#    --server.headless true: no intenta abrir un navegador
#    (no hay navegador en la máquina de Colab).
proceso = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"),
    stderr=subprocess.STDOUT,
)

# 3. Dar tiempo al servidor a arrancar
time.sleep(6)

# 4. Abrir el túnel
url = ngrok.connect(8501)
print("=" * 60)
print(f"  Tu aplicación está en: {url.public_url}")
print("=" * 60)
print("\nSi ves una pantalla de advertencia de ngrok, pulsa 'Visit Site'.")

## Paso 5. Ver los errores

Si la aplicación no carga o muestra un error, los mensajes de Streamlit están
en `streamlit.log`, porque el servidor corre en segundo plano y no imprime
en la salida de las celdas.


In [ ]:
!tail -30 streamlit.log

## Paso 6. Detener todo

Ejecuta esta celda cuando termines, o antes de volver a lanzar el paso 4.


In [ ]:
ngrok.kill()
proceso.terminate()
print("Túnel cerrado y servidor detenido")

---

## Alternativas a ngrok

### Método nativo de Colab (sin registro)

Genera un enlace que **solo funciona para ti**, en tu sesión. No sirve para
compartir, pero es perfecto para desarrollar sin depender de ngrok:

```python
from google.colab import output
output.serve_kernel_port_as_website(8501)
```

### localtunnel (sin registro)

```python
!npm install -g localtunnel
!streamlit run app.py --server.port 8501 &>/dev/null &
!curl https://loca.lt/mytunnelpassword    # esta es la contraseña que pedirá
!npx localtunnel --port 8501
```

---

## Limitaciones de ngrok con cuenta gratuita

| Limitación | Detalle |
|---|---|
| Un solo túnel simultáneo | Por eso el `ngrok.kill()` inicial |
| URL aleatoria | Cambia en cada reinicio, salvo dominio estático |
| Página de advertencia | Los visitantes ven una pantalla intermedia |
| Depende de Colab | La URL muere al cerrar la sesión |

### Dominio estático (opcional)

ngrok ofrece un dominio fijo gratuito por cuenta. Créalo en el panel
(**Domains**) y úsalo así:

```python
url = ngrok.connect(8501, domain="tu-dominio.ngrok-free.app")
```

---

## Siguiente paso: despliegue permanente

Para una URL que no dependa de Colab, consulta el **módulo 9** del curso:
subir el código a GitHub y desplegarlo en Streamlit Community Cloud.
El resultado es una dirección `https://tu-app.streamlit.app` permanente
y gratuita.
